# 🚀 Model Deployment with Flask

> **Deploy machine learning models as web services**

This notebook demonstrates how to deploy trained ML models as REST APIs using Flask, making them accessible for real-world applications.

## 🎯 Learning Objectives

By the end of this notebook, you will:
- **Create** Flask web services for ML models
- **Handle** model serialization and loading
- **Implement** proper error handling and validation
- **Test** deployed models with real requests
- **Monitor** model performance in production

In [ ]:
import numpy as np
import pandas as pd
import pickle
import joblib
import json
from flask import Flask, request, jsonify
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import requests
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("✅ All imports successful!")

## 🤖 Model Training and Serialization

In [ ]:
# Create and train a sample model
print("🔧 Training sample model...")

# Generate sample data
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5, 
                          n_redundant=2, n_clusters_per_class=1, random_state=42)

feature_names = [f'feature_{i}' for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=feature_names)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate model
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.4f}")

# Save model and scaler
joblib.dump(model, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(feature_names, 'feature_names.pkl')

print("✅ Model trained and saved!")

## 🌐 Flask Web Service Implementation

In [ ]:
# Create Flask application code
flask_app_code = '''
import numpy as np
import pandas as pd
import joblib
import json
from flask import Flask, request, jsonify
from datetime import datetime
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize Flask app
app = Flask(__name__)

# Load model artifacts
try:
    model = joblib.load('model.pkl')
    scaler = joblib.load('scaler.pkl')
    feature_names = joblib.load('feature_names.pkl')
    logger.info("Model artifacts loaded successfully")
except Exception as e:
    logger.error(f"Error loading model artifacts: {e}")
    model = None
    scaler = None
    feature_names = None

class ModelPredictor:
    def __init__(self, model, scaler, feature_names):
        self.model = model
        self.scaler = scaler
        self.feature_names = feature_names
        self.prediction_count = 0
    
    def validate_input(self, data):
        """Validate input data"""
        if not isinstance(data, dict):
            return False, "Input must be a JSON object"
        
        if 'features' not in data:
            return False, "Missing 'features' key in input"
        
        features = data['features']
        
        if not isinstance(features, dict):
            return False, "Features must be a dictionary"
        
        # Check if all required features are present
        missing_features = set(self.feature_names) - set(features.keys())
        if missing_features:
            return False, f"Missing features: {list(missing_features)}"
        
        # Check if all values are numeric
        for feature, value in features.items():
            if not isinstance(value, (int, float)):
                return False, f"Feature '{feature}' must be numeric"
        
        return True, "Valid input"
    
    def preprocess_input(self, features):
        """Preprocess input features"""
        # Create DataFrame with correct feature order
        feature_values = [features[name] for name in self.feature_names]
        X = np.array(feature_values).reshape(1, -1)
        
        # Scale features
        X_scaled = self.scaler.transform(X)
        
        return X_scaled
    
    def predict(self, data):
        """Make prediction"""
        # Validate input
        is_valid, message = self.validate_input(data)
        if not is_valid:
            return {'error': message}, 400
        
        try:
            # Preprocess input
            X_scaled = self.preprocess_input(data['features'])
            
            # Make prediction
            prediction = self.model.predict(X_scaled)[0]
            probability = self.model.predict_proba(X_scaled)[0]
            
            # Update prediction count
            self.prediction_count += 1
            
            # Prepare response
            response = {
                'prediction': int(prediction),
                'probability': {
                    'class_0': float(probability[0]),
                    'class_1': float(probability[1])
                },
                'confidence': float(max(probability)),
                'timestamp': datetime.now().isoformat(),
                'model_version': '1.0'
            }
            
            logger.info(f"Prediction made: {prediction} (confidence: {max(probability):.3f})")
            
            return response, 200
            
        except Exception as e:
            logger.error(f"Prediction error: {e}")
            return {'error': f'Prediction failed: {str(e)}'}, 500

# Initialize predictor
if model and scaler and feature_names:
    predictor = ModelPredictor(model, scaler, feature_names)
else:
    predictor = None

@app.route('/health', methods=['GET'])
def health_check():
    """Health check endpoint"""
    if predictor:
        return jsonify({
            'status': 'healthy',
            'model_loaded': True,
            'predictions_made': predictor.prediction_count,
            'timestamp': datetime.now().isoformat()
        })
    else:
        return jsonify({
            'status': 'unhealthy',
            'model_loaded': False,
            'error': 'Model not loaded'
        }), 500

@app.route('/predict', methods=['POST'])
def predict():
    """Prediction endpoint"""
    if not predictor:
        return jsonify({'error': 'Model not loaded'}), 500
    
    try:
        data = request.get_json()
        if not data:
            return jsonify({'error': 'No JSON data provided'}), 400
        
        result, status_code = predictor.predict(data)
        return jsonify(result), status_code
        
    except Exception as e:
        logger.error(f"Request processing error: {e}")
        return jsonify({'error': 'Request processing failed'}), 500

@app.route('/model-info', methods=['GET'])
def model_info():
    """Model information endpoint"""
    if not predictor:
        return jsonify({'error': 'Model not loaded'}), 500
    
    return jsonify({
        'model_type': 'RandomForestClassifier',
        'features': predictor.feature_names,
        'num_features': len(predictor.feature_names),
        'model_version': '1.0',
        'predictions_made': predictor.prediction_count
    })

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)
'''

# Save Flask app to file
with open('app.py', 'w') as f:
    f.write(flask_app_code)

print("✅ Flask application code saved to 'app.py'")
print("\n📝 To run the Flask app:")
print("   python app.py")
print("\n🌐 The API will be available at:")
print("   http://localhost:5000")

## 🧪 Testing the Deployed Model

In [ ]:
# Create test client functions
def test_health_endpoint(base_url='http://localhost:5000'):
    """Test health check endpoint"""
    try:
        response = requests.get(f'{base_url}/health')
        print(f"Health Check Status: {response.status_code}")
        print(f"Response: {response.json()}")
        return response.status_code == 200
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to Flask app. Make sure it's running.")
        return False

def test_prediction_endpoint(base_url='http://localhost:5000'):
    """Test prediction endpoint"""
    # Create sample input
    sample_features = {f'feature_{i}': np.random.randn() for i in range(10)}
    
    test_data = {
        'features': sample_features
    }
    
    try:
        response = requests.post(
            f'{base_url}/predict',
            json=test_data,
            headers={'Content-Type': 'application/json'}
        )
        
        print(f"Prediction Status: {response.status_code}")
        print(f"Response: {response.json()}")
        return response.status_code == 200
        
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to Flask app. Make sure it's running.")
        return False

def test_model_info_endpoint(base_url='http://localhost:5000'):
    """Test model info endpoint"""
    try:
        response = requests.get(f'{base_url}/model-info')
        print(f"Model Info Status: {response.status_code}")
        print(f"Response: {response.json()}")
        return response.status_code == 200
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to Flask app. Make sure it's running.")
        return False

def run_comprehensive_tests(base_url='http://localhost:5000'):
    """Run comprehensive API tests"""
    print("🧪 Running comprehensive API tests...\n")
    
    # Test 1: Health check
    print("=== Test 1: Health Check ===")
    health_ok = test_health_endpoint(base_url)
    print()
    
    # Test 2: Model info
    print("=== Test 2: Model Info ===")
    info_ok = test_model_info_endpoint(base_url)
    print()
    
    # Test 3: Valid prediction
    print("=== Test 3: Valid Prediction ===")
    pred_ok = test_prediction_endpoint(base_url)
    print()
    
    # Test 4: Invalid input
    print("=== Test 4: Invalid Input ===")
    try:
        invalid_data = {'invalid': 'data'}
        response = requests.post(
            f'{base_url}/predict',
            json=invalid_data,
            headers={'Content-Type': 'application/json'}
        )
        print(f"Invalid Input Status: {response.status_code}")
        print(f"Response: {response.json()}")
        invalid_ok = response.status_code == 400
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to Flask app.")
        invalid_ok = False
    
    print()
    
    # Summary
    all_tests = [health_ok, info_ok, pred_ok, invalid_ok]
    passed = sum(all_tests)
    total = len(all_tests)
    
    print(f"=== Test Summary ===")
    print(f"Tests passed: {passed}/{total}")
    
    if passed == total:
        print("🎉 All tests passed! Your API is working correctly.")
    else:
        print("⚠️ Some tests failed. Check the Flask app.")

print("✅ Test functions defined!")
print("\n📝 To test the API (after starting Flask app):")
print("   run_comprehensive_tests()")

## 🎯 Practice Problems

### **Problem 1: Model Monitoring**
Add monitoring capabilities to track model performance.

In [ ]:
class ModelMonitor:
    def __init__(self):
        """
        Initialize model monitoring system
        
        Track:
        - Prediction latency
        - Prediction distribution
        - Input feature statistics
        - Error rates
        """
        # Your code here
        pass
    
    def log_prediction(self, features, prediction, probability, latency):
        """Log prediction details"""
        # Your code here
        pass
    
    def get_monitoring_report(self):
        """Generate monitoring report"""
        # Your code here
        pass

# Test your implementation
# monitor = ModelMonitor()
# Add monitoring to Flask app

### **Problem 2: Batch Prediction Endpoint**
Add support for batch predictions.

In [ ]:
def create_batch_prediction_endpoint():
    """
    Create Flask endpoint for batch predictions
    
    Features:
    - Accept multiple samples in one request
    - Return predictions for all samples
    - Handle large batches efficiently
    - Validate all inputs
    
    Returns:
    str: Flask route code
    """
    # Your code here
    pass

# Test your implementation
# batch_endpoint_code = create_batch_prediction_endpoint()
# print(batch_endpoint_code)

## 🎯 Key Takeaways

1. **Flask** provides a simple way to deploy ML models as web services
2. **Input validation** is crucial for robust production systems
3. **Error handling** ensures graceful failure modes
4. **Monitoring** helps track model performance in production
5. **Testing** validates API functionality before deployment

## 🔗 Next Steps

1. **Complete the practice problems** above
2. **Add authentication and rate limiting**
3. **Move to the next notebook**: Docker Containerization

---

**Excellent deployment skills!** 🎉 You can now deploy ML models as production-ready web services.